In [2]:
#import libraries
import os
import pandas as pd

In [3]:
caminho = r'.\DB Trains\behindertenparken.csv'

df = pd.read_csv(caminho, sep=';', encoding='utf-8')
print(f"This dataset has {df.shape[0]} rows and {df.shape[1]} columns.")

df.info()


This dataset has 1874 rows and 18 columns.
<class 'pandas.DataFrame'>
RangeIndex: 1874 entries, 0 to 1873
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   source              1874 non-null   str    
 1   original_uid        1717 non-null   str    
 2   osm_id              77 non-null     str    
 3   purpose             1874 non-null   str    
 4   name                1565 non-null   str    
 5   operator_name       895 non-null    str    
 6   address             996 non-null    str    
 7   type                683 non-null    str    
 8   park_and_ride_type  626 non-null    str    
 9   lat                 1874 non-null   float64
 10  lon                 1874 non-null   float64
 11  capacity            1616 non-null   float64
 12  max_stay            25 non-null     str    
 13  has_lighting        34 non-null     str    
 14  opening_hours       466 non-null    str    
 15  description         329

In [4]:
df.head()

,source,original_uid,osm_id,purpose,name,operator_name,address,type,park_and_ride_type,lat,lon,capacity,max_stay,has_lighting,opening_hours,description,public_url,photo_url
0,Landratsamt Calw,INFRA-de:08235:1510-PARKPLATZ-1,NaN,CAR,Rathaus (Gechingen),NaN,NaN,NaN,NO,48.694964,8.829307,1.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Landratsamt Calw,INFRA-de:08235:10169-PARKPLATZ-1,NaN,CAR,Burgschule,NaN,NaN,NaN,NO,48.552753,8.724225,2.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Landratsamt Calw,INFRA-de:08235:10112-PARKPLATZ-1,NaN,CAR,Altensteiger Str.,NaN,NaN,NaN,YES,48.549395,8.714575,2.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Landratsamt Calw,INFRA-de:08235:10449-PARKPLATZ-1,NaN,CAR,Uferstr.,NaN,NaN,NaN,YES,48.549411,8.714552,2.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Landratsamt Emmendingen,INFRA-de:08316:102-PARKPLATZ-1,NaN,CAR,Sport- und Familienbad,NaN,NaN,NaN,YES,48.227785,8.167884,3.0,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# 1. What is the total capacity and the average number of disabled parking spaces in these parking lots?
print("--- Capacity Analysis ---")
print(f"Disabled Parking Spaces: {df['capacity'].sum()}")
print(f"Average Spaces per Lot: {df['capacity'].mean():.2f}")
print(f"Largest Lot has: {df['capacity'].max()} spaces\n")

# 2. What are the main operators of these parking lots? (Top 5)
print("--- Top 5 Parking Lot Operators ---")
print(df['operator_name'].value_counts().head(5))

--- Capacity Analysis ---
Disabled Parking Spaces: 5591.0
Average Spaces per Lot: 3.46
Largest Lot has: 150.0 spaces

--- Top 5 Parking Lot Operators ---
operator_name
Stadt Karlsruhe                          260
Stadt Heidelberg                         202
Contipark Parkgaragengesellschaft mbH     96
Stadt Pforzheim                           73
Stadt Ulm                                 48
Name: count, dtype: int64


In [6]:
# 1. Let's see the 5 largest parking lots and where they are located
print("--- TOP 5 Parking Lots ---")
print(df[['name', 'capacity', 'operator_name']].sort_values(by='capacity', ascending=False).head(5))

# 2. What are the main types of parking lots? (Top 5)
print("\n--- Main Types of Parking Lots ---")
print(df['type'].value_counts(dropna=False))

# 3. What is the purpose of these parking lots? (Top 5)
print("\n--- Purpose / Objective of Each Lot ---")
print(df['purpose'].value_counts())

--- TOP 5 Parking Lots ---
                                      name  capacity operator_name
1496                München, Allianz Arena     150.0           NaN
1497                    Munich, Neue Messe     120.0           NaN
1462     Schönefeld, Berlin Airport P3 BER     113.0           NaN
1461  Schönefeld, Berlin Airport P7/P8 BER     108.0           NaN
1674     Schönefeld, Berlin Airport P1 BER      81.0           NaN

--- Main Types of Parking Lots ---
type
NaN                          1191
ON_STREET                     256
CAR_PARK                      167
UNDERGROUND                   139
OFF_STREET_PARKING_GROUND     121
Name: count, dtype: int64

--- Purpose / Objective of Each Lot ---
purpose
CAR    1874
Name: count, dtype: int64


In [ ]:
# extract city from column 'name' which contains the city name followed by a comma and the parking lot name

df['city'] = df['name'].str.split(',', expand=True)[0].str.strip()


# Create a new column 'capacity_size' to categorize the parking lots based on their capacity 
# Less than 10 slots = Small | Between 10 and 50 slots = Medium | More than 50 slots = Large
def categorize_capacity(capacity):
    if pd.isna(capacity):
        return 'Not Specified'
    elif capacity < 10:
        return 'Small (< 10 slots)'
    elif capacity <= 50:
        return 'Medium (10-50 slots)'
    else:
        return 'Large (> 50 slots)'

# Aplicando a função na coluna 'capacity' para criar a nova coluna 'capacity_size'
df['capacity_size'] = df['capacity'].apply(categorize_capacity)


# Top 5 cities with the most mapped parking lots for disabled people and their total capacity
print("--- TOP 5 CITIES ---")
print(df.groupby('city')['capacity'].sum().sort_values(ascending=False).head(5))

print("\n--- DISTRIBUTION OF PARKINGLOTS BY SIZE ---")
print(df['capacity_size'].value_counts())

--- TOP 5 CITIES ---
city
Schönefeld    354.0
Stuttgart     294.0
Munich        252.0
Düsseldorf    227.0
Hamburg       221.0
Name: capacity, dtype: float64

--- DISTRIBUTION OF PARKINGLOTS BY SIZE ---
capacity_size
Small (< 10 slots)      1515
Not Specified            258
Medium (10-50 slots)      92
Large (> 50 slots)         9
Name: count, dtype: int64


In [11]:
#read punctuality dataset
try:
    df_punc = pd.read_csv(r'.\DB Trains\puenktlichkeit.csv', sep=';', encoding='utf-8')
except UnicodeDecodeError:
    df_punc = pd.read_csv(r'.\DB Trains\puenktlichkeit.csv', sep=';', encoding='latin1')

# Verifying the first few rows of the punctuality dataset
print("--- COLUMNS OF THE PUNCTUALITY DATASET ---")
print(df_punc.columns)

print("\n--- OVERVIEW OF THE DATA ---")
display(df_punc.head()) # If not in Jupyter, use print(df_punc.head())

print("\n--- INFORMATION AND DATA TYPES ---")
print(df_punc.info())

--- COLUMNS OF THE PUNCTUALITY DATASET ---
Index(['linie', 'puenktlichkeitsniveau_an', 'jahr', 'monat'], dtype='str')

--- OVERVIEW OF THE DATA ---


,linie,puenktlichkeitsniveau_an,jahr,monat
0,RB 61 Itzehoe - Hamburg (Hbf),"91,81",2010,1
1,RB 62 Heide - Itzehoe,"92,84",2010,1
2,RB 64 St.-Peter-Ording - Husum,"98,15",2010,1
3,RB 71 Wrist - Hamburg-Altona,"87,57",2010,1
4,RB 73 Eckernförde - Kiel,"89,94",2010,1



--- INFORMATION AND DATA TYPES ---
<class 'pandas.DataFrame'>
RangeIndex: 4339 entries, 0 to 4338
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   linie                     4339 non-null   str  
 1   puenktlichkeitsniveau_an  4339 non-null   str  
 2   jahr                      4339 non-null   int64
 3   monat                     4339 non-null   int64
dtypes: int64(2), str(2)
memory usage: 135.7 KB
None


In [12]:
# Cleaning the punctuality column to convert it to float for analysis
df_punc['puenktlichkeitsniveau_an'] = (
    df_punc['puenktlichkeitsniveau_an']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# Avg punctuality of all lines
print(f"Average punctuality: {df_punc['puenktlichkeitsniveau_an'].mean():.2f}%")

# 3. Top 5 lines with the highest and lowest punctuality
print("\n--- TOP 5 MOST PUNCTUAL LINES ---")
print(df_punc.groupby('linie')['puenktlichkeitsniveau_an'].mean().sort_values(ascending=False).head(5))

print("\n--- TOP 5 LINES WITH THE WORST PUNCTUALITY ---")
print(df_punc.groupby('linie')['puenktlichkeitsniveau_an'].mean().sort_values().head(5))

Average punctuality: 92.79%

--- TOP 5 MOST PUNCTUAL LINES ---
linie
A 3 Elmshorn - Ulzburg Süd          99.086119
RB 66 Esbjerg - Tønder - Niebüll    98.840825
A 2 Ulzburg Süd - Norderstedt       98.793507
RB 64 St.-Peter-Ording - Husum      98.636943
A 1 Neumünster - Eidelstedt         97.580821
Name: puenktlichkeitsniveau_an, dtype: float64

--- TOP 5 LINES WITH THE WORST PUNCTUALITY ---
linie
RE 7 Flensburg - Hamburg (Hbf)         82.905793
RE 7 Flensburg/Kiel - Hamburg (Hbf)    83.696944
RE 1 Hamburg (Hbf) - Schwerin          83.919108
RE 70 Kiel - Hamburg (Hbf)             84.193885
RE 6 Westerland - Hamburg-Altona       84.811146
Name: puenktlichkeitsniveau_an, dtype: float64


In [22]:
import re

# Cities thas has parkinglots
cities_unique = df['city'].dropna().unique()

# list to store the matches
matches = []

# cities with parking lots and trains passing through them
for cidade in cities_unique:
    cidade_limpa = str(cidade).strip()
    if len(cidade_limpa) < 3: # Ignora textos muito curtos para evitar falsos positivos
        continue
        
    # filtering the punctuality dataset for trains that pass through the city
    padrao_seguro = re.escape(cidade_limpa)
    
    trens_na_cidade = df_punc[df_punc['linie'].str.contains(padrao_seguro, case=False, na=False)]
    
    if not trens_na_cidade.empty:
        media_pontualidade = trens_na_cidade['puenktlichkeitsniveau_an'].mean()
        
        dados_estacionamento = df[df['city'] == cidade]
        total_vagas = dados_estacionamento['capacity'].sum()
        total_locais = dados_estacionamento['name'].count()
        
        matches.append({
            'city': cidade_limpa,
            'average_punctuality': round(media_pontualidade, 2),
            'total_disabled_parking_spaces': int(total_vagas),
            'count_parking_lots': int(total_locais)
        })

# Final Dataframe
df_insight = pd.DataFrame(matches)

# Order the DataFrame by average punctuality in ascending order
df_insight = df_insight.sort_values(by='average_punctuality', ascending=True).reset_index(drop=True)
df_insight['average_punctuality'] = df_insight['average_punctuality'].astype(str) + '%'

print(f" total Cities: {len(df_insight)}")
print("\n--- BUSINESS INSIGHT: TRAIN PUNCTUALITY VS ACCESSIBILITY ---")
display(df_insight.head(10))

 total Cities: 5

--- BUSINESS INSIGHT: TRAIN PUNCTUALITY VS ACCESSIBILITY ---


,city,average_punctuality,total_disabled_parking_spaces,count_parking_lots
0,Flensburg,86.59%,5,1
1,Hamburg,88.48%,221,22
2,Lübeck,94.0%,13,1
3,Norderstedt,98.79%,8,1
4,Elmshorn,99.09%,5,1


In [23]:
# Adjusting column names for better readability
df_insight.columns = [
    'city', 
    'avg_punctuality_pct', 
    'total_disabled_parking_spaces', 
    'count_parking_lots'
]

# Final display of the processed dataset for analysis
print("--- FINAL DATASET FOR ANALYSIS ---")
display(df_insight)

--- FINAL DATASET FOR ANALYSIS ---


,city,avg_punctuality_pct,total_disabled_parking_spaces,count_parking_lots
0,Flensburg,86.59%,5,1
1,Hamburg,88.48%,221,22
2,Lübeck,94.0%,13,1
3,Norderstedt,98.79%,8,1
4,Elmshorn,99.09%,5,1
